# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world - calling APIs, querying databases, or running code - to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to a **local OpenAI-compatible model endpoint** using the **Open AI Framework**.
2. Give the agent a **tool** - a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.


## Setup

Before running this notebook, make sure you have:

1. **A local model server** exposing an OpenAI-compatible Chat Completions API (for example, Ollama with `http://localhost:11434/v1`).
2. **A local model pulled and running** (for example, `ollama pull llama3.2`).
3. **Set the required environment variables:**
   - `LOCAL_MODEL_BASE_URL` - local API base URL (must include `/v1`).
   - `LOCAL_MODEL_ID` - model name served by your local endpoint (for example, `llama3.2`).
   - `LOCAL_MODEL_API_KEY` - optional API key (use `local` if your server does not require auth).

The cell below installs the Python packages you need.


In [ ]:
%pip install agent-framework python-dotenv -q


In [15]:
import logging
logging.getLogger("agent_framework.openai").setLevel(logging.ERROR)

import os
from typing import Annotated

from dotenv import load_dotenv
from agent_framework import tool
from agent_framework.openai import OpenAIChatCompletionClient

load_dotenv()

local_model_base_url = os.getenv("LOCAL_MODEL_BASE_URL", "http://localhost:11434/v1")
local_model_id = os.getenv("LOCAL_MODEL_ID", "llama3.2")
local_model_api_key = os.getenv("LOCAL_MODEL_API_KEY", "local")

chat_client = OpenAIChatCompletionClient(
    base_url=local_model_base_url,
    api_key=local_model_api_key,
    model=local_model_id,
)


## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [16]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
from agent_framework import Agent

instructions = (
    "You are a helpful travel agent. Help users find their perfect vacation "
    "destination based on their preferences. Use the get_destinations tool "
    "to see available destinations."
)

agent = Agent(
    client=chat_client,
    tools=[get_destinations],
    name="TravelAgent",
    instructions=instructions,
)

user_prompt = "I'm looking for a warm beach destination. What do you recommend?"
response = await agent.run(user_prompt)

print(response)

## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [17]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk.text or "", end="", flush=True)


Tokyo is a fascinating travel destination that offers a unique blend of traditional and modern culture. As the capital city of Japan, Tokyo is a bustling metropolis that seamlessly merges ancient temples, shrines, and gardens with cutting-edge technology, fashion, and food.

Must-visit attractions in Tokyo include:

1. **Shibuya Crossing**: Known for its busiest intersection in the world, Shibuya Crossing is a spectacle to behold.
2. **Tokyo Skytree**: At 634 meters tall, Tokyo Skytree offers breathtaking views of the city from its observation decks.
3. **Tsukiji Fish Market**: While the inner market has moved to a new location, the outer market still offers a fascinating glimpse into Japan's seafood culture.
4. **Meiji Shrine**: Dedicated to the deified spirits of Emperor Meiji and his wife, Empress Shoken, this shrine is a serene oasis in the midst of the bustling city.
5. **Asakusa**: This historic district is home to Senso-ji Temple, one of the oldest and most colorful temples in J

## Summary

In this lesson you learned how to:

- **Create a chat client** connected to a local OpenAI-compatible model endpoint via `OpenAIChatCompletionClient`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.
